!pip install pysindy

In [5]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from pysindy.feature_library import PolynomialLibrary, GeneralizedLibrary
from pysindy import SINDy
import pysindy as ps


In [6]:
def lorenz(t, z):
    return [10*(z[1] - z[0]),
            z[0]*(28 - z[2]) - z[1],
            z[0]*z[1] - 8/3*z[2]]

In [9]:
t_span = (0, 2)
t_eval = np.arange(0, 2, 0.002)
initial_conditions = [-8, 8, 27]
poly_order = 2
threshold = 0.1
# Solve the differential equation
sol = solve_ivp(lorenz, t_span, initial_conditions, t_eval=t_eval)

# Extract the solution at the evaluated time points
x = sol.y.T
optimizer=ps.STLSQ(threshold=threshold)
alpha = 2.0  # Example value for alpha
def f1(x, alpha=alpha):
    return alpha * x

def f2(x, y, alpha=alpha):
    return alpha * x * y

functions = [
    lambda x: x,
    lambda x, y: x * y,
    lambda x: f1(x),
    lambda x, y: f2(x, y),
    lambda x: 1  # Constant term
]

# Proceed with SINDy modeling
model = ps.SINDy(
        optimizer,
        feature_library=ps.CustomLibrary(library_functions=functions),
        feature_names=['x','y','z']
    )


In [114]:
model.fit(x, t=t_eval[1]-t_eval[0])
model.print()

(x)' = -10.004 f0(x) + 10.003 f0(y)
(y)' = 27.886 f0(x) + -0.982 f0(y) + -0.995 f1(x,z)
(z)' = -2.666 f0(z) + 0.999 f1(x,y)
